Célula 1 - Importações e Carga dos Dados da PRF

In [7]:
import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações iniciais
sns.set_theme(style="whitegrid")

# 1. Carregar todos os arquivos da PRF do Pará
caminho_prf = '../data/raw/prf/*pa*.csv' 
lista_dfs_prf = [pd.read_csv(arq, sep=',', encoding='utf-8', low_memory=False) for arq in glob.glob(caminho_prf)]
df_prf = pd.concat(lista_dfs_prf, ignore_index=True)

# 2. Padronização Robusta de Colunas
# Tudo para minúsculo e troca espaços por underline
df_prf.columns = df_prf.columns.str.lower().str.replace(' ', '_')

# REMOÇÃO DE ACENTOS (Garante que "condição" vire "condicao")
df_prf.columns = df_prf.columns.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')

# Remove colunas fantasma (lixo do delimitador)
df_prf = df_prf.loc[:, ~df_prf.columns.str.contains('^unnamed')]

print(f"Base da PRF carregada e blindada contra erros de encoding. Formato: {df_prf.shape}")

# 1. Identifica todas as colunas que contêm texto (tipo 'object')
colunas_de_texto = df_prf.select_dtypes(include=['object']).columns

# 2. Varre todas essas colunas aplicando a remoção de acentos e passando para minúsculo
for col in colunas_de_texto:
    # Ignoramos a coluna de horário para não quebrar a formatação de 12:30:00
    if col != 'horario': 
        df_prf[col] = (df_prf[col]
                       .astype(str)
                       .str.normalize('NFKD')
                       .str.encode('ascii', errors='ignore')
                       .str.decode('utf-8')
                       .str.lower()
                       .str.strip()) # Remove espaços invisíveis no começo e fim da palavra

print("Acentos removidos e textos padronizados dentro da base de dados.")

Base da PRF carregada e blindada contra erros de encoding. Formato: (19664, 37)


C:\Users\Aspire 3 Intel i3\AppData\Local\Temp\ipykernel_2840\210346606.py:29: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colunas_de_texto = df_prf.select_dtypes(include=['object']).columns


Acentos removidos e textos padronizados dentro da base de dados.


Célula 2 - Tratamento de Tipos e Outliers da PRF

In [8]:
# 1. Correção de coordenadas numéricas (Vrgula para Ponto)
for col in ['latitude', 'longitude']:
    if col in df_prf.columns and df_prf[col].dtype == 'object':
        df_prf[col] = df_prf[col].str.replace(',', '.').astype(float)

# 2. Criação de Features Temporais Básicas
if 'data_inversa' in df_prf.columns:
    df_prf['data_inversa'] = pd.to_datetime(df_prf['data_inversa'], format='%Y-%m-%d', errors='coerce')
    df_prf['ano'] = df_prf['data_inversa'].dt.year
    df_prf['mes'] = df_prf['data_inversa'].dt.month

if 'horario' in df_prf.columns:
    df_prf['hora_decimal'] = pd.to_datetime(df_prf['horario'], format='%H:%M:%S', errors='coerce').dt.hour

# 3. Tratamento de Outliers no KM via IQR
df_prf['km'] = df_prf['km'].astype(str).str.replace(',', '.')
df_prf['km'] = pd.to_numeric(df_prf['km'], errors='coerce')
km_valido = df_prf['km'].dropna()

Q1 = km_valido.quantile(0.25)
Q3 = km_valido.quantile(0.75)
IQR = Q3 - Q1
limite_superior = Q3 + 1.5 * IQR

# Filtrando a base removendo inconsistências
df_prf_limpo = df_prf[(df_prf['km'].notna()) & (df_prf['km'] >= 0) & (df_prf['km'] <= limite_superior)].copy()

# 4. Remoção de colunas administrativas inuteis para o ML
colunas_inuteis_prf = ['id', 'pesid', 'id_veiculo', 'regional', 'delegacia', 'uop', 'ordem_tipo_acidente']
colunas_drop = [col for col in colunas_inuteis_prf if col in df_prf_limpo.columns]
df_prf_limpo = df_prf_limpo.drop(columns=colunas_drop)

# 5. Exportação do Checkpoint da PRF
os.makedirs('../data/processed', exist_ok=True)
df_prf_limpo.to_csv('../data/processed/prf_limpo.csv', index=False, sep=';', encoding='utf-8')
print(f"Base da PRF exportada com sucesso! Linhas restando: {len(df_prf_limpo)}")

Base da PRF exportada com sucesso! Linhas restando: 18052
